# 7.4 边缘AI生产监控与可观测性

模型部署上线只是开始，生产环境的持续监控和可观测性才是保障服务质量的基础设施。边缘AI的监控比云端更复杂：设备碎片化、网络不稳定、数据难以上报。本章建立边缘AI监控的完整方法论。

## 为什么边缘AI需要专门的监控？

| 云端监控 | 边缘AI监控 | 差异 |
|---------|-----------|------|
| 稳定网络 | 不稳定/离线 | 数据上报延迟或丢失 |
| 统一硬件 | 碎片化硬件 | 指标含义不同 |
| 全量日志 | 采样上报 | 带宽限制，只能上报摘要 |
| 实时告警 | 延迟告警 | 离线期间无法告警 |
| 远程控制 | 有限控制 | OTA更新有延迟 |
| 丰富上下文 | 有限上下文 | 隐私限制，不能上传用户数据 |

## 7.4.1 边缘AI监控指标体系

### 三层指标体系

```
┌─────────────────────────────────────────────────┐
│  L1: 业务指标 (用户可感知)                        │
│  → 首token延迟、生成速度、用户满意度              │
├─────────────────────────────────────────────────┤
│  L2: 模型指标 (模型质量)                          │
│  → 输出质量、PPL、token分布、拒绝率               │
├─────────────────────────────────────────────────┤
│  L3: 系统指标 (硬件/运行时)                       │
│  → 内存使用、CPU/NPU利用率、温度、功耗             │
└─────────────────────────────────────────────────┘
```

### 关键指标定义

| 层级 | 指标 | 采集方式 | 告警阈值 | 产业标准 |
|------|------|---------|---------|----------|
| L1 | TTFT | 推理计时 | >1000ms | <500ms |
| L1 | ITL | 推理计时 | >100ms | <50ms |
| L1 | 吞吐 | 推理计时 | <10 tok/s | >15 tok/s |
| L2 | 输出长度异常 | 输出分析 | >4x输入长度 | - |
| L2 | 重复率 | 输出分析 | >30%重复 | <10% |
| L2 | 拒绝/安全过滤率 | 内容审核 | >5% | <1% |
| L3 | 内存峰值 | 系统API | >设备80% | <70% |
| L3 | NPU温度 | 传感器 | >85°C | <80°C |
| L3 | 崩溃率 | 异常捕获 | >1% | <0.1% |
| L3 | 降级触发率 | 运行时 | >10% | <5% |

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import numpy as np

@dataclass
class MetricSample:
    timestamp: float
    ttft_ms: float
    itl_ms: float
    memory_mb: float
    temperature_c: float
    npu_utilization: float
    degradation_level: int
    error: Optional[str] = None

class EdgeMonitor:
    """边缘AI监控器"""
    ALERT_RULES = {
        'ttft_high': {'metric': 'ttft_ms', 'threshold': 1000, 'severity': 'warning'},
        'itl_high': {'metric': 'itl_ms', 'threshold': 100, 'severity': 'warning'},
        'memory_high': {'metric': 'memory_mb', 'threshold': 4000, 'severity': 'critical'},
        'temperature_high': {'metric': 'temperature_c', 'threshold': 85, 'severity': 'critical'},
        'degradation': {'metric': 'degradation_level', 'threshold': 2, 'severity': 'warning'},
        'error': {'metric': 'error', 'threshold': None, 'severity': 'critical'},
    }

    def __init__(self, device_id, upload_interval_s=300):
        self.device_id = device_id
        self.upload_interval = upload_interval_s
        self.samples: List[MetricSample] = []
        self.alerts: List[Dict] = []
        self.aggregated = {}

    def collect(self, sample: MetricSample):
        self.samples.append(sample)
        self._check_alerts(sample)

    def _check_alerts(self, sample: MetricSample):
        for rule_name, rule in self.ALERT_RULES.items():
            if rule['threshold'] is None:
                if sample.error:
                    self.alerts.append({'rule': rule_name, 'severity': rule['severity'],
                                        'value': sample.error, 'timestamp': sample.timestamp})
            else:
                val = getattr(sample, rule['metric'], None)
                if val is not None and val > rule['threshold']:
                    self.alerts.append({'rule': rule_name, 'severity': rule['severity'],
                                        'value': val, 'threshold': rule['threshold'],
                                        'timestamp': sample.timestamp})

    def aggregate_for_upload(self):
        if not self.samples:
            return None
        ttfts = [s.ttft_ms for s in self.samples]
        itls = [s.itl_ms for s in self.samples]
        memories = [s.memory_mb for s in self.samples]
        temps = [s.temperature_c for s in self.samples]
        errors = [s for s in self.samples if s.error]
        degraded = [s for s in self.samples if s.degradation_level >= 2]
        self.aggregated = {
            'device_id': self.device_id,
            'n_samples': len(self.samples),
            'ttft': {'p50': np.percentile(ttfts, 50), 'p90': np.percentile(ttfts, 90),
                     'p99': np.percentile(ttfts, 99)},
            'itl': {'p50': np.percentile(itls, 50), 'p90': np.percentile(itls, 90),
                    'p99': np.percentile(itls, 99)},
            'memory_peak_mb': max(memories),
            'temperature_max_c': max(temps),
            'error_rate': len(errors) / len(self.samples),
            'degradation_rate': len(degraded) / len(self.samples),
            'alert_count': len(self.alerts),
        }
        self.samples = []
        self.alerts = []
        return self.aggregated

monitor = EdgeMonitor('device_001', upload_interval_s=300)
np.random.seed(42)
for i in range(100):
    temp = 65 + np.random.exponential(5)
    if temp > 80:
        itl = np.random.normal(80, 15)
        degrad = 2
    else:
        itl = np.random.normal(45, 8)
        degrad = 0
    error = 'npu_timeout' if np.random.random() < 0.01 else None
    sample = MetricSample(
        timestamp=i * 10,
        ttft_ms=np.random.normal(250, 30),
        itl_ms=max(itl, 10),
        memory_mb=np.random.normal(3200, 100),
        temperature_c=temp,
        npu_utilization=np.random.uniform(0.02, 0.15),
        degradation_level=degrad,
        error=error
    )
    monitor.collect(sample)

report = monitor.aggregate_for_upload()
print("=== 边缘AI监控聚合报告 ===")
print(f"设备: {report['device_id']}")
print(f"采样数: {report['n_samples']}")
print(f"\n--- L1 业务指标 ---")
print(f"TTFT: P50={report['ttft']['p50']:.0f}ms, P90={report['ttft']['p90']:.0f}ms, P99={report['ttft']['p99']:.0f}ms")
print(f"ITL:  P50={report['itl']['p50']:.0f}ms, P90={report['itl']['p90']:.0f}ms, P99={report['itl']['p99']:.0f}ms")
print(f"\n--- L3 系统指标 ---")
print(f"内存峰值: {report['memory_peak_mb']:.0f} MB")
print(f"最高温度: {report['temperature_max_c']:.1f} °C")
print(f"错误率: {report['error_rate']:.1%}")
print(f"降级率: {report['degradation_rate']:.1%}")
print(f"告警数: {report['alert_count']}")

## 7.4.2 边缘AI的数据上报策略

### 三种上报策略

| 策略 | 数据量 | 实时性 | 适用场景 |
|------|--------|--------|----------|
| **全量上报** | 高 | 实时 | 开发调试期 |
| **聚合上报** | 低(1/100) | 延迟5-15分钟 | 生产监控 |
| **异常触发上报** | 极低 | 事件驱动 | 关键告警 |

### 产业实践：聚合+异常混合上报

```
正常数据: 每5分钟聚合一次 → 上报摘要(P50/P90/P99)
异常数据: 立即上报 → 崩溃/超时/降级事件
离线缓存: 网络不可用时本地缓存 → 恢复后批量上报
```

### 数据上报的隐私考量

- **不上报用户输入/输出**：仅上报性能指标，不上报模型内容
- **差分隐私**：聚合数据添加噪声，防止逆向推断用户行为
- **本地优先**：尽可能在端侧完成分析，仅上报异常摘要

## 7.4.3 告警与故障响应

### 告警分级

| 级别 | 触发条件 | 响应时间 | 通知方式 |
|------|---------|---------|----------|
| **P0 Critical** | 崩溃率>1%, 服务不可用 | <15分钟 | 电话+短信+IM |
| **P1 Warning** | 延迟P99>2x基线, 降级率>10% | <1小时 | 短信+IM |
| **P2 Info** | 延迟P90>1.5x基线, 温度>80°C | <4小时 | IM |
| **P3 Notice** | 新设备型号首次上线, SDK版本变更 | 下一工作日 | 邮件 |

### 故障响应SOP

```
1. 告警触发 → 确认影响范围(单设备/批量/全量)
2. 影响评估 → 用户影响比例、业务损失估算
3. 止血 → 自动降级/回滚/熔断
4. 根因分析 → 日志分析+复现+定位
5. 修复 → 代码修复/配置调整/模型回滚
6. 验证 → 灰度验证修复效果
7. 复盘 → 记录故障时间线、根因、改进措施
```

### 自动化故障响应

| 故障类型 | 自动响应 | 人工介入 |
|---------|---------|----------|
| **NPU崩溃** | 自动切换CPU推理 | 分析崩溃日志 |
| **内存溢出** | 自动减少上下文长度 | 优化内存使用 |
| **热节流** | 自动降低推理频率 | 优化模型/硬件 |
| **精度异常** | 自动回滚到上一版本 | 分析精度退化原因 |
| **SDK不兼容** | 自动降级到兼容版本 | 更新SDK适配 |

In [ ]:
class FleetMonitor:
    """设备集群监控仪表盘"""
    def __init__(self):
        self.device_reports: Dict[str, Dict] = {}

    def ingest(self, device_id, report):
        self.device_reports[device_id] = report

    def fleet_dashboard(self):
        if not self.device_reports:
            return "无数据"
        all_ttfts = []
        all_itls = []
        total_errors = 0
        total_samples = 0
        total_degraded = 0
        device_models = {}
        for did, r in self.device_reports.items():
            all_ttfts.append(r['ttft']['p50'])
            all_itls.append(r['itl']['p50'])
            total_errors += r['error_rate'] * r['n_samples']
            total_samples += r['n_samples']
            total_degraded += r['degradation_rate'] * r['n_samples']
        return {
            'n_devices': len(self.device_reports),
            'total_samples': total_samples,
            'fleet_ttft_p50_ms': np.mean(all_ttfts),
            'fleet_itl_p50_ms': np.mean(all_itls),
            'fleet_error_rate': total_errors / total_samples if total_samples > 0 else 0,
            'fleet_degradation_rate': total_degraded / total_samples if total_samples > 0 else 0,
        }

fleet = FleetMonitor()
device_models = ['SM-S928B', 'SM-S928B', 'iPhone16,2', 'iPhone16,2',
                  'Pixel 8 Pro', 'SM-A556B', 'iPad14,5']
np.random.seed(42)
for i, model in enumerate(device_models):
    fleet.ingest(f'device_{i:03d}_{model}', {
        'device_id': f'device_{i:03d}_{model}',
        'n_samples': np.random.randint(50, 200),
        'ttft': {'p50': np.random.normal(250, 40), 'p90': np.random.normal(350, 50),
                 'p99': np.random.normal(500, 80)},
        'itl': {'p50': np.random.normal(50, 15), 'p90': np.random.normal(70, 20),
                'p99': np.random.normal(100, 30)},
        'memory_peak_mb': np.random.normal(3000, 500),
        'temperature_max_c': np.random.normal(70, 10),
        'error_rate': np.random.uniform(0, 0.02),
        'degradation_rate': np.random.uniform(0, 0.1),
        'alert_count': np.random.randint(0, 5),
    })

dashboard = fleet.fleet_dashboard()
print("=== 设备集群监控仪表盘 ===")
print(f"在线设备数: {dashboard['n_devices']}")
print(f"总采样数: {dashboard['total_samples']}")
print(f"\n--- 集群性能概览 ---")
print(f"TTFT P50: {dashboard['fleet_ttft_p50_ms']:.0f} ms")
print(f"ITL P50:  {dashboard['fleet_itl_p50_ms']:.0f} ms")
print(f"错误率:   {dashboard['fleet_error_rate']:.2%}")
print(f"降级率:   {dashboard['fleet_degradation_rate']:.1%}")

print(f"\n=== 边缘AI监控总结 ===")
print(f"1. 三层指标: L1业务(用户体验) + L2模型(输出质量) + L3系统(硬件状态)")
print(f"2. 聚合上报: 每5分钟上报P50/P90/P99摘要，异常事件立即上报")
print(f"3. 隐私优先: 不上报用户输入/输出，仅上报性能指标")
print(f"4. 告警分级: P0(崩溃)>P1(性能)>P2(趋势)>P3(信息)")
print(f"5. 自动响应: NPU崩溃→CPU降级, 内存溢出→减上下文, 精度异常→回滚")
print(f"6. 集群监控: 聚合多设备数据，发现系统性问题和硬件差异")